In [1]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import os
from datetime import datetime

class EquipmentReconApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Equipment Reconciliation Tool")
        self.root.geometry("600x550")
        
        # Styles for a cleaner look
        style = ttk.Style()
        style.configure("TButton", padding=6, relief="flat", background="#ccc")
        style.configure("TLabel", padding=5, font=("Helvetica", 10))

        # Variables to store file paths
        self.source_path = tk.StringVar()
        self.master_path = tk.StringVar()
        self.export_folder = tk.StringVar()

        # --- GUI LAYOUT ---
        # Title
        title_label = ttk.Label(root, text="Equipment Data Reconciliation", font=("Helvetica", 14, "bold"))
        title_label.pack(pady=15)

        # Container for form inputs
        form_frame = ttk.Frame(root, padding=20)
        form_frame.pack(fill='both', expand=True)

        # 1. Source File Selection
        ttk.Label(form_frame, text="Source File (UCSD EQUIPMENT Summary):").grid(row=0, column=0, sticky='w')
        ttk.Entry(form_frame, textvariable=self.source_path, width=50).grid(row=1, column=0, padx=5, pady=5)
        ttk.Button(form_frame, text="Browse", command=self.select_source).grid(row=1, column=1, padx=5)

        # 2. Master File Selection
        ttk.Label(form_frame, text="Master File (Part Numbers Map):").grid(row=2, column=0, sticky='w', pady=(10, 0))
        ttk.Entry(form_frame, textvariable=self.master_path, width=50).grid(row=3, column=0, padx=5, pady=5)
        ttk.Button(form_frame, text="Browse", command=self.select_master).grid(row=3, column=1, padx=5)

        # 3. Export Folder Selection
        ttk.Label(form_frame, text="Export Folder:").grid(row=4, column=0, sticky='w', pady=(10, 0))
        ttk.Entry(form_frame, textvariable=self.export_folder, width=50).grid(row=5, column=0, padx=5, pady=5)
        ttk.Button(form_frame, text="Browse", command=self.select_folder).grid(row=5, column=1, padx=5)

        # 4. Action Buttons
        btn_frame = ttk.Frame(root, padding=20)
        btn_frame.pack(fill='x')
        self.run_btn = tk.Button(btn_frame, text="RUN RECONCILIATION", bg="#4CAF50", fg="white", 
                                 font=("Helvetica", 10, "bold"), height=2, command=self.run_process)
        self.run_btn.pack(fill='x')

        # Status Bar
        self.status_var = tk.StringVar()
        self.status_var.set("Ready")
        status_bar = tk.Label(root, textvariable=self.status_var, bd=1, relief=tk.SUNKEN, anchor='w')
        status_bar.pack(side=tk.BOTTOM, fill=tk.X)

    # --- HELPER FUNCTIONS ---

    def select_source(self):
        filename = filedialog.askopenfilename(title="Select Source File", filetypes=[("Excel Files", "*.xlsx *.xls")])
        if filename: self.source_path.set(filename)

    def select_master(self):
        filename = filedialog.askopenfilename(title="Select Master File", filetypes=[("Excel Files", "*.xlsx *.xls")])
        if filename: self.master_path.set(filename)

    def select_folder(self):
        folder = filedialog.askdirectory(title="Select Export Folder")
        if folder: self.export_folder.set(folder)

    # --- CORE LOGIC ---

    def run_process(self):
        # 1. Validation
        if not all([self.source_path.get(), self.master_path.get(), self.export_folder.get()]):
            messagebox.showerror("Error", "Please select both input files and an export folder.")
            return

        try:
            self.status_var.set("Processing data...")
            self.root.update_idletasks() # Force UI update        

            # =========================================================
            # PART 1: Process Tab 1 (Index 0) - Reconciliation Logic
            # =========================================================
            try:
                # sheet_name=0 loads the first sheet
                # header=16 (Row 17) is correct for Tab 1
                df_source = pd.read_excel(self.source_path.get(), sheet_name=0, header=16, dtype={'ACCOUNT NO.': str})
            except Exception as e:
                messagebox.showerror("Error", f"Could not read the FIRST sheet of the Source File.\nDetails: {e}")
                self.status_var.set("Error: Read Failed")
                return
            
            df_master = pd.read_excel(self.master_path.get())

            # --- Stop at First Null (Tab 1) ---
            if 'ACCOUNT NO.' not in df_source.columns:
                 raise ValueError(f"Column 'ACCOUNT NO.' not found in Tab 1. Found: {df_source.columns.tolist()}")

            null_indices = df_source.index[df_source['ACCOUNT NO.'].isna()]
            if not null_indices.empty:
                df_source = df_source.iloc[:null_indices[0]]

            # --- Cleaning (Tab 1) ---
            df_source['ACCOUNT NO.'] = df_source['ACCOUNT NO.'].fillna('').astype(str).str.strip().str.lstrip('0')
            
            initial_count = len(df_source)
            df_source.drop_duplicates(inplace=True)
            print(f"Tab 1: Removed {initial_count - len(df_source)} fully duplicate rows.")

            # Prepare for Merge
            df_source['EQUIP_CLEAN'] = df_source['EQUIP'].astype(str).str.strip()
            df_master['Equipment_CLEAN'] = df_master['Equipment'].astype(str).str.strip()

            # --- Merge (Tab 1) ---
            df_merged = pd.merge(
                df_source, 
                df_master[['Equipment_CLEAN', 'Part No.']], 
                left_on='EQUIP_CLEAN', 
                right_on='Equipment_CLEAN', 
                how='left'
            )

            # --- Formatting (Tab 1) ---
            rename_map = {'Qty': 'Quantity', 'QTY': 'Quantity', 'qty': 'Quantity'}
            df_merged.rename(columns=rename_map, inplace=True)
            
            df_merged['Part No.'] = df_merged['Part No.'].fillna('NO MATCH FOUND')
            # Ensure no trailing decimals and pad with zeros
            df_merged['Part No.'] = df_merged['Part No.'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(4)

            final_columns = ['ACCOUNT NO.', 'EQUIP', 'Quantity', 'Part No.']
            missing_cols = [c for c in final_columns if c not in df_merged.columns]
            if missing_cols:
                raise ValueError(f"Missing columns in Tab 1 data: {missing_cols}")

            df_final = df_merged[final_columns].copy()

            # --- Split Logic (Tab 1) - UPDATED BY PYTHON DEV ---
            
            # 1. Initial Split based on Count
            account_counts = df_final['ACCOUNT NO.'].value_counts()
            df_final['Freq_Count'] = df_final['ACCOUNT NO.'].map(account_counts)
            
            # Use .copy() to avoid SettingWithCopyWarning later
            df_unique = df_final[df_final['Freq_Count'] == 1].drop(columns=['Freq_Count']).copy()
            df_duplicates = df_final[df_final['Freq_Count'] >= 2].drop(columns=['Freq_Count']).copy()

            # 2. CHANGE: Remove rows from duplicates where EQUIP contains "Under the Counter Unit"
            # Using case=False to be robust (matches "under the counter unit" and "Under the Counter Unit")
            df_duplicates = df_duplicates[~df_duplicates['EQUIP'].astype(str).str.contains("Under the Counter Unit", case=False, na=False)]

            # 3. CHANGE: Re-evaluate Uniqueness
            # After removing the rows above, some Account Numbers might now only appear once.
            # We need to move those to df_unique.
            
            dup_counts_recalc = df_duplicates['ACCOUNT NO.'].value_counts()
            df_duplicates['New_Count'] = df_duplicates['ACCOUNT NO.'].map(dup_counts_recalc)

            # Rows that are now unique (Count == 1)
            rows_to_move = df_duplicates[df_duplicates['New_Count'] == 1].drop(columns=['New_Count'])
            
            # Rows that are still duplicates (Count >= 2)
            df_duplicates_final = df_duplicates[df_duplicates['New_Count'] >= 2].drop(columns=['New_Count'])

            # Move the newly unique rows to the unique dataframe
            if not rows_to_move.empty:
                df_unique = pd.concat([df_unique, rows_to_move], ignore_index=True)

            # Update df_duplicates to only contain the remaining duplicates
            df_duplicates = df_duplicates_final.sort_values(by='ACCOUNT NO.')

            # =========================================================
            # PART 2: Process Tab 3 (Index 2) - STD Import Logic
            # =========================================================
            try:
                # FIX: header=15 (Row 16) for Tab 3 because the structure differs slightly
                df_std = pd.read_excel(self.source_path.get(), sheet_name=2, header=15, dtype={'ACCOUNT NO.': str})
            except Exception as e:
                print(f"Warning: Could not read 3rd sheet (Index 2). Skipping STD Import.\n{e}")
                df_std = pd.DataFrame() 

            if not df_std.empty:
                if 'ACCOUNT NO.' in df_std.columns:
                    # --- Stop at First Null (Tab 3) ---
                    null_indices_std = df_std.index[df_std['ACCOUNT NO.'].isna()]
                    if not null_indices_std.empty:
                        df_std = df_std.iloc[:null_indices_std[0]]

                    # --- Cleaning (Tab 3) ---
                    df_std['ACCOUNT NO.'] = df_std['ACCOUNT NO.'].fillna('').astype(str).str.strip().str.lstrip('0')
                    df_std.rename(columns=rename_map, inplace=True) # Rename QTY -> Quantity

                    # --- Merge for Part # ---
                    # NOTE: Merging on 'Equipment' because 'Account No.' does not exist in the Master File.
                    if 'EQUIP' in df_std.columns:
                        df_std['EQUIP_CLEAN'] = df_std['EQUIP'].astype(str).str.strip()
                        
                        df_std_merged = pd.merge(
                            df_std, 
                            df_master[['Equipment_CLEAN', 'Part No.']], 
                            left_on='EQUIP_CLEAN', 
                            right_on='Equipment_CLEAN', 
                            how='left'
                        )
                        
                        # Clean Part No (zfill 4)
                        df_std_merged['Part No.'] = df_std_merged['Part No.'].fillna('NO MATCH FOUND')
                        df_std_merged['Part No.'] = df_std_merged['Part No.'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(4)
                        
                        # Rename to 'Part #' as requested
                        df_std_merged.rename(columns={'Part No.': 'Part #'}, inplace=True)
                    else:
                        raise ValueError("Column 'EQUIP' not found in Tab 3, cannot merge for Part #.")

                    # --- Group By Logic ---
                    if 'Quantity' in df_std_merged.columns:
                        df_std_merged['Quantity'] = pd.to_numeric(df_std_merged['Quantity'], errors='coerce').fillna(0)
                        
                        # Group by Account NO. AND Part # so we don't lose the part info
                        df_std_grouped = df_std_merged.groupby(['ACCOUNT NO.', 'Part #'])['Quantity'].sum().reset_index()
                    else:
                         raise ValueError("Column 'Quantity' not found in 3rd Tab.")
                else:
                    raise ValueError(f"Column 'ACCOUNT NO.' not found in 3rd Tab. Found: {df_std.columns.tolist()}")
            else:
                df_std_grouped = pd.DataFrame()

            # =========================================================
            # EXPORT
            # =========================================================
            timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
            
            # File 1: The Standard Reconciliation
            file_name_1 = f"UCSD EQUIPMENT Equipment Import {timestamp}.xlsx"
            full_path_1 = os.path.join(self.export_folder.get(), file_name_1)

            with pd.ExcelWriter(full_path_1, engine='openpyxl') as writer:
                df_unique.to_excel(writer, sheet_name='Equipment Import', index=False)
                df_duplicates.to_excel(writer, sheet_name='Multiple Occurrences', index=False)

            # File 2: The Grouped STD Import (With Part #)
            if not df_std_grouped.empty:
                file_name_2 = f"UCSD EQUIPMENT STD Import {timestamp}.xlsx"
                full_path_2 = os.path.join(self.export_folder.get(), file_name_2)
                df_std_grouped.to_excel(full_path_2, index=False)
                success_msg = f"Process Complete!\n\nFile 1 Created:\n{file_name_1}\n\nFile 2 Created:\n{file_name_2}"
            else:
                success_msg = f"Process Complete!\n\nFile 1 Created:\n{file_name_1}\n\n(Warning: 3rd Tab empty or skipped, File 2 not created.)"

            self.status_var.set("Ready")
            messagebox.showinfo("Success", success_msg)

        except Exception as e:
            self.status_var.set("Error")
            messagebox.showerror("Execution Error", f"An error occurred:\n{str(e)}")
            print(e)

if __name__ == "__main__":
    root = tk.Tk()
    app = EquipmentReconApp(root)
    root.mainloop()

c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Tab 1: Removed 4 fully duplicate rows.
